In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install pandas scikit-learn pyarrow

In [ ]:
# %cd /content/drive/MyDrive/projects/Quantum-Echos/mimic-cxs_data
# !wget -r -N -c -np --user sebasmos777 --ask-password https://physionet.org/files/mimiciv/3.1/hosp/admissions.csv.gz
# Unzip the file
# !gunzip /content/drive/MyDrive/projects/Quantum-Echos/mimic-cxs_data/physionet.org/files/mimiciv/3.1/hosp/admissions.csv.gz
# !mv /content/drive/MyDrive/projects/Quantum-Echos/mimic-cxs_data/physionet.org/files/mimiciv/3.1/hosp/admissions.csv /content/drive/MyDrive/projects/Quantum-Echos/mimic-cxs_data/

# Dataset design - 1 seed

**IMPORTANT**: This notebook now saves as `.pkl` (pickle) instead of `.csv` to preserve full multi-dimensional embeddings.

The CSV format was truncating embeddings from 88,064 features to ~216 features due to numpy's string representation.
Parquet also cannot handle multi-dimensional arrays (only 1D), so pickle is used to preserve the original shape (8, 8, 1376).

**NOTE**: This version uses **INSURANCE** as the label instead of gender. Insurance labels are derived from `admisions.csv`:
- **Private** → Class 1 (one-hot: [0., 1.])
- **Medicaid/Medicare** → Class 0 (one-hot: [1., 0.])

In [ ]:
import numpy as np
import pandas as pd
from glob import glob
import os
from tqdm import tqdm
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                            roc_auc_score, confusion_matrix, classification_report)
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

CONFIG = {
    'SAMPLE_SIZE': 2000,
    'TRAIN_TEST_SPLIT_RATIO': 0.8,
    'RANDOM_STATE': 42,

    'EMBEDDING_PATH': "/content/drive/MyDrive/projects/Quantum-Echos/mimic-cxs_data/ve_cxr/elixir/batch_*.npz",
    'METADATA_PATH': '/content/drive/MyDrive/projects/Quantum-Echos/mimic-cxs_data/verified_dicom_clincalinfo.csv',
    'METADATA_EXTENDED_PATH': "/content/drive/MyDrive/projects/Quantum-Echos/mimic-cxs_data/verified_dicom_updatedlabel_wide_processed.csv",
    'PATH_TO_ADMISIONS': "/content/drive/MyDrive/projects/Quantum-Echos/mimic-cxs_data/admissions.csv",

    # Output directory - all results saved here (NO subfolders created)
    'OUTPUT_DIR': '/content/drive/MyDrive/projects/Quantum-Echos/mimic-cxs_data/data-cleaned',

    'PATIENT_ID_COLUMN': 'subject_id',
    'INSURANCE_COLUMN': 'new_insurance_type',  # Will be created from admissions insurance
    'DICOM_PATH_COLUMN': 'dicom_path',

    # SAMPLING_STRATEGY determines output filename automatically
    'SAMPLING_STRATEGY': 1,

    'REMOVE_DUPLICATES': True,
    'REMOVE_DUPLICATE_FILENAMES': True,
    'ONE_IMAGE_PER_PATIENT': True,

    'STRIP_FILE_EXTENSION': True,
    'DEBUG_FILENAME_MATCHING': False,
    'VERBOSE': True
}

SAMPLING_STRATEGIES = {
    1: {
        'name': 'Balanced Insurance (50/50)',
        'description': 'Ensures equal distribution across insurance types (Private vs Medicaid/Medicare)',
        'function': 'strategy_1_balanced_insurance'
    },
    2: {
        'name': 'Stratified Proportional',
        'description': 'Maintains original insurance proportions',
        'function': 'strategy_2_stratified_proportional'
    },
    3: {
        'name': 'Simple Random Sampling',
        'description': 'Pure random selection without stratification',
        'function': 'strategy_3_random_sampling'
    },
    4: {
        'name': 'Stratified by Pathology Status',
        'description': 'Preserves original distribution of normal vs abnormal cases',
        'function': 'strategy_4_pathology_stratified'
    },
    5: {
        'name': 'Stratified by Pathology Status (Excluding Support Devices Only)',
        'description': 'Preserves distribution, excludes cases with only Support Devices and No Finding',
        'function': 'strategy_5_pathology_stratified_clean'
    }
}

# Auto-generate output filename based on sampling strategy
OUTPUT_NAME = f"data_type{CONFIG['SAMPLING_STRATEGY']}_insurance.pkl"

# Create output directory (single directory, no subfolders)
os.makedirs(CONFIG['OUTPUT_DIR'], exist_ok=True)

# Setup logging to file
log_messages = []
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_filename = f"preprocessing_log_strategy{CONFIG['SAMPLING_STRATEGY']}_insurance_{timestamp}.txt"
log_path = os.path.join(CONFIG['OUTPUT_DIR'], log_filename)

def log_print(message):
    """Print message and store for later saving to file"""
    log_messages.append(message)
    if CONFIG['VERBOSE']:
        print(message)

def save_log():
    """Save all log messages to file"""
    with open(log_path, 'w') as f:
        f.write('\n'.join(log_messages))
    print(f"\nLog saved to: {log_path}")

log_print("=" * 80)
log_print("CHEST X-RAY INSURANCE CLASSIFICATION PIPELINE")
log_print("=" * 80)
log_print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
log_print(f"Target Sample Size: {CONFIG['SAMPLE_SIZE']:,} samples")
log_print(f"Train/Test split: {CONFIG['TRAIN_TEST_SPLIT_RATIO']:.0%} / {1-CONFIG['TRAIN_TEST_SPLIT_RATIO']:.0%}")
log_print(f"Sampling Strategy: {CONFIG['SAMPLING_STRATEGY']} - {SAMPLING_STRATEGIES[CONFIG['SAMPLING_STRATEGY']]['name']}")
log_print(f"Output directory: {CONFIG['OUTPUT_DIR']}")
log_print(f"Output file: {OUTPUT_NAME} (auto-generated based on strategy {CONFIG['SAMPLING_STRATEGY']})")
log_print("=" * 80)

log_print("\n" + "=" * 80)
log_print("STEP 1: LOADING COMPLETE METADATA")
log_print("=" * 80)

meta = pd.read_csv(CONFIG['METADATA_PATH'])

log_print(f"Loaded metadata from: {CONFIG['METADATA_PATH']}")
log_print(f"TOTAL ROWS IN FULL DATASET: {len(meta):,}")
log_print(f"Original columns: {list(meta.columns)[:10]}...")

log_print("\nCreating filename column from dicom_path...")
meta['filename'] = meta[CONFIG['DICOM_PATH_COLUMN']].apply(lambda x: os.path.basename(x) if pd.notna(x) else None)

if CONFIG['STRIP_FILE_EXTENSION']:
    meta['filename'] = meta['filename'].apply(lambda x: os.path.splitext(x)[0] if pd.notna(x) else None)
    log_print("Stripped file extensions from filenames to match NPZ format")

meta = meta[meta['filename'].notna()].reset_index(drop=True)
log_print(f"Created filename column from {CONFIG['DICOM_PATH_COLUMN']}")
log_print(f"Rows after removing invalid paths: {len(meta):,}")
log_print(f"Example filename: {meta['filename'].iloc[0]}")

# ============================================================================
# STEP 1.5: LOADING AND MERGING ADMISSIONS DATA FOR INSURANCE
# ============================================================================
log_print("\n" + "=" * 80)
log_print("STEP 1.5: LOADING AND PREPROCESSING ADMISSIONS DATA FOR INSURANCE")
log_print("=" * 80)

admissions = pd.read_csv(CONFIG['PATH_TO_ADMISIONS'])
log_print(f"Loaded admissions from: {CONFIG['PATH_TO_ADMISIONS']}")
log_print(f"Total admissions rows: {len(admissions):,}")
log_print(f"Admissions columns: {list(admissions.columns)}")

# ============================================================================
# INSURANCE PREPROCESSING - DETAILED LOGGING
# ============================================================================
log_print("\n" + "-" * 60)
log_print("INSURANCE COLUMN ANALYSIS (RAW DATA)")
log_print("-" * 60)

# Show raw insurance values before any processing
raw_insurance_counts = admissions['insurance'].value_counts(dropna=False)
log_print(f"\nRaw insurance values in admissions.csv:")
for ins_val, count in raw_insurance_counts.items():
    percentage = (count / len(admissions)) * 100
    log_print(f"   '{ins_val}': {count:,} ({percentage:.2f}%)")

log_print(f"\nTotal unique raw insurance values: {admissions['insurance'].nunique()}")
log_print(f"Missing/NaN insurance values: {admissions['insurance'].isna().sum():,}")

# ============================================================================
# INSURANCE MAPPING LOGIC (MATCHING REFERENCE NOTEBOOK)
# ============================================================================
log_print("\n" + "-" * 60)
log_print("INSURANCE MAPPING LOGIC")
log_print("-" * 60)
log_print("\nMapping rules (same as reference MIMIC_raw dataset class):")
log_print("   'Private'                    -> 'Private'        (Class 1: one-hot [0., 1.])")
log_print("   'Medicaid' or 'Medicare'     -> 'Medicaid_Medicare' (Class 0: one-hot [1., 0.])")
log_print("   Other/Unknown/NaN            -> EXCLUDED (set to None)")

def map_insurance_type(insurance):
    """
    Map insurance values to binary categories matching reference notebook:
    - Private -> "Private" (will be encoded as [0., 1.])
    - Medicaid/Medicare -> "Medicaid_Medicare" (will be encoded as [1., 0.])
    - Other/Unknown -> None (excluded from dataset)
    """
    if pd.isna(insurance):
        return None
    insurance_str = str(insurance).strip()
    if insurance_str.lower() == 'private':
        return 'Private'
    elif insurance_str.lower() in ['medicaid', 'medicare']:
        return 'Medicaid_Medicare'
    else:
        return None  # Exclude other/unknown insurance types

# Apply mapping
admissions['new_insurance_type'] = admissions['insurance'].apply(map_insurance_type)

# ============================================================================
# POST-MAPPING ANALYSIS
# ============================================================================
log_print("\n" + "-" * 60)
log_print("POST-MAPPING ANALYSIS")
log_print("-" * 60)

# Count how many were excluded
excluded_count = admissions['new_insurance_type'].isna().sum()
included_count = admissions['new_insurance_type'].notna().sum()
log_print(f"\nRecords after mapping:")
log_print(f"   Included (valid insurance): {included_count:,} ({included_count/len(admissions)*100:.2f}%)")
log_print(f"   Excluded (other/unknown):   {excluded_count:,} ({excluded_count/len(admissions)*100:.2f}%)")

# Show which raw values were excluded
excluded_values = admissions[admissions['new_insurance_type'].isna()]['insurance'].value_counts(dropna=False)
if len(excluded_values) > 0:
    log_print(f"\nExcluded raw insurance values:")
    for val, count in excluded_values.items():
        log_print(f"   '{val}': {count:,} records excluded")

# Clean and deduplicate
admissions_clean = admissions[admissions['new_insurance_type'].notna()][['subject_id', 'new_insurance_type']].copy()
admissions_clean = admissions_clean.drop_duplicates(subset=['subject_id'], keep='first')

# ============================================================================
# FINAL CLASS DISTRIBUTION
# ============================================================================
log_print("\n" + "-" * 60)
log_print("FINAL INSURANCE CLASS DISTRIBUTION")
log_print("-" * 60)

num_classes = admissions_clean['new_insurance_type'].nunique()
class_names = sorted(admissions_clean['new_insurance_type'].unique())
log_print(f"\nNumber of classes: {num_classes}")
log_print(f"Class names: {class_names}")

log_print(f"\nUnique patients with valid insurance: {len(admissions_clean):,}")
ins_counts = admissions_clean['new_insurance_type'].value_counts()
for ins_type, count in ins_counts.items():
    percentage = (count / len(admissions_clean)) * 100
    log_print(f"   {ins_type}: {count:,} ({percentage:.2f}%)")

# ============================================================================
# ONE-HOT ENCODING REFERENCE
# ============================================================================
log_print("\n" + "-" * 60)
log_print("ONE-HOT ENCODING REFERENCE (for model training)")
log_print("-" * 60)
log_print("\nEncoding scheme (matching reference notebook MIMIC_raw class):")
log_print("   'Private'          -> torch.Tensor([0., 1.])  # Class 1")
log_print("   'Medicaid_Medicare' -> torch.Tensor([1., 0.])  # Class 0")
log_print("\nThis matches the reference code:")
log_print("   if insurance_type == 'Private':")
log_print("       output_insurance = torch.Tensor([0., 1.])")
log_print("   elif insurance_type == 'Medicaid' or insurance_type == 'Medicare':")
log_print("       output_insurance = torch.Tensor([1., 0.])")

# ============================================================================
# MERGE WITH METADATA
# ============================================================================
log_print("\n" + "-" * 60)
log_print("MERGING WITH IMAGE METADATA")
log_print("-" * 60)

meta_before_merge = len(meta)
meta = pd.merge(meta, admissions_clean, on='subject_id', how='inner')
log_print(f"\nMetadata rows before merge: {meta_before_merge:,}")
log_print(f"Metadata rows after merge:  {len(meta):,}")
log_print(f"Rows lost in merge: {meta_before_merge - len(meta):,} (patients without valid insurance)")

log_print(f"\nInsurance distribution in FULL merged dataset:")
insurance_counts = meta[CONFIG['INSURANCE_COLUMN']].value_counts()
for insurance, count in insurance_counts.items():
    percentage = (count / len(meta)) * 100
    log_print(f"   {insurance}: {count:,} ({percentage:.2f}%)")

unique_patients = meta[CONFIG['PATIENT_ID_COLUMN']].nunique()
log_print(f"\nTotal unique patients: {unique_patients:,}")
log_print(f"Average images per patient: {len(meta) / unique_patients:.2f}")

def remove_duplicates(df, patient_col, random_state):
    log_print("\n" + "=" * 80)
    log_print("STEP 2: REMOVING DUPLICATES - KEEPING ONLY UNIQUE DATA")
    log_print("=" * 80)

    original_size = len(df)
    log_print(f"Original dataset size: {original_size:,}")

    if CONFIG['REMOVE_DUPLICATE_FILENAMES']:
        log_print("\nRemoving duplicate filenames...")
        df_unique = df.drop_duplicates(subset=['filename'], keep='first')
        log_print(f"After removing duplicate filenames: {len(df_unique):,} rows")
        log_print(f"   Removed {original_size - len(df_unique):,} duplicate filenames")
    else:
        df_unique = df.copy()

    if CONFIG['ONE_IMAGE_PER_PATIENT']:
        log_print(f"\nKeeping only ONE image per patient (using {patient_col})...")
        patients_before = df_unique[patient_col].nunique()
        df_unique = df_unique.groupby(patient_col).sample(n=1, random_state=random_state).reset_index(drop=True)
        patients_after = df_unique[patient_col].nunique()

        log_print(f"Unique patients before: {patients_before:,}")
        log_print(f"Unique patients after: {patients_after:,}")
        log_print(f"Final unique dataset size: {len(df_unique):,} rows")
        log_print(f"Each row now represents a UNIQUE patient")

    log_print(f"\nDEDUPLICATION COMPLETE!")
    log_print(f"Reduced from {original_size:,} to {len(df_unique):,} UNIQUE samples")
    log_print(f"Reduction: {(1 - len(df_unique)/original_size)*100:.2f}%")

    log_print(f"\nInsurance distribution in unique dataset:")
    unique_insurance = df_unique[CONFIG['INSURANCE_COLUMN']].value_counts()
    for insurance, count in unique_insurance.items():
        percentage = (count / len(df_unique)) * 100
        log_print(f"   {insurance}: {count:,} ({percentage:.2f}%)")

    return df_unique

if CONFIG['REMOVE_DUPLICATES']:
    meta_unique = remove_duplicates(meta, CONFIG['PATIENT_ID_COLUMN'], CONFIG['RANDOM_STATE'])
else:
    meta_unique = meta.copy()
    log_print("\nSkipping deduplication (REMOVE_DUPLICATES=False)")

def strategy_1_balanced_insurance(df, target_size, random_state, insurance_col):
    log_print("\n" + "=" * 80)
    log_print(f"STEP 3: SAMPLING STRATEGY 1 - BALANCED INSURANCE (50/50)")
    log_print("=" * 80)
    log_print(f"Target: {target_size:,} samples ({target_size//2:,} per insurance type)")
    log_print(f"Available: {len(df):,} unique samples")

    df_clean = df[df[insurance_col].notna()].copy()
    log_print(f"Removed {len(df) - len(df_clean):,} rows with missing insurance values")
    log_print(f"Clean dataset: {len(df_clean):,} samples")

    samples_per_insurance = target_size // 2
    sampled_dfs = []

    for insurance_value in sorted(df_clean[insurance_col].unique()):
        log_print(f"\nSampling insurance: {insurance_value}")
        insurance_df = df_clean[df_clean[insurance_col] == insurance_value]
        log_print(f"   Available {insurance_value}: {len(insurance_df):,}")

        if len(insurance_df) >= samples_per_insurance:
            insurance_sample = insurance_df.sample(n=samples_per_insurance, random_state=random_state)
            log_print(f"   Sampled {samples_per_insurance:,} for {insurance_value}")
        else:
            insurance_sample = insurance_df
            log_print(f"   Only {len(insurance_sample):,} available for {insurance_value}")

        sampled_dfs.append(insurance_sample)

    df_sampled = pd.concat(sampled_dfs, ignore_index=True)

    log_print(f"\nFinal sample size: {len(df_sampled):,}")
    log_print(f"Insurance distribution:")
    for insurance, count in df_sampled[insurance_col].value_counts().items():
        log_print(f"   {insurance}: {count:,} ({count/len(df_sampled)*100:.2f}%)")

    return df_sampled

def strategy_2_stratified_proportional(df, target_size, random_state, insurance_col):
    log_print("\n" + "=" * 80)
    log_print(f"STEP 3: SAMPLING STRATEGY 2 - STRATIFIED PROPORTIONAL")
    log_print("=" * 80)
    log_print(f"Target: {target_size:,} samples")
    log_print(f"Available: {len(df):,} unique samples")
    log_print(f"Maintains original insurance proportions")

    df_clean = df[df[insurance_col].notna()].copy()
    log_print(f"Removed {len(df) - len(df_clean):,} rows with missing insurance values")
    log_print(f"Clean dataset: {len(df_clean):,} samples")

    insurance_proportions = df_clean[insurance_col].value_counts(normalize=True)
    log_print(f"\nOriginal proportions:")
    for insurance, prop in insurance_proportions.items():
        log_print(f"   {insurance}: {prop*100:.2f}%")

    sampled_dfs = []
    for insurance_value in df_clean[insurance_col].unique():
        insurance_df = df_clean[df_clean[insurance_col] == insurance_value]
        n_samples = int(target_size * len(insurance_df) / len(df_clean))
        if len(insurance_df) >= n_samples:
            sampled_dfs.append(insurance_df.sample(n=n_samples, random_state=random_state))
        else:
            sampled_dfs.append(insurance_df)

    df_sampled = pd.concat(sampled_dfs, ignore_index=True)

    log_print(f"\nFinal sample size: {len(df_sampled):,}")
    log_print(f"Final insurance distribution:")
    for insurance, count in df_sampled[insurance_col].value_counts().items():
        log_print(f"   {insurance}: {count:,} ({count/len(df_sampled)*100:.2f}%)")

    return df_sampled

def strategy_3_random_sampling(df, target_size, random_state, insurance_col):
    log_print("\n" + "=" * 80)
    log_print(f"STEP 3: SAMPLING STRATEGY 3 - SIMPLE RANDOM SAMPLING")
    log_print("=" * 80)
    log_print(f"Target: {target_size:,} samples")
    log_print(f"Available: {len(df):,} unique samples")
    log_print(f"Pure random selection (no stratification)")

    df_clean = df[df[insurance_col].notna()].copy()
    log_print(f"Removed {len(df) - len(df_clean):,} rows with missing insurance values")
    log_print(f"Clean dataset: {len(df_clean):,} samples")

    if len(df_clean) >= target_size:
        df_sampled = df_clean.sample(n=target_size, random_state=random_state).reset_index(drop=True)
        log_print(f"\nSampled {target_size:,} random samples")
    else:
        df_sampled = df_clean
        log_print(f"\nOnly {len(df_clean):,} samples available")

    log_print(f"\nFinal sample size: {len(df_sampled):,}")
    log_print(f"Insurance distribution:")
    for insurance, count in df_sampled[insurance_col].value_counts().items():
        log_print(f"   {insurance}: {count:,} ({count/len(df_sampled)*100:.2f}%)")

    return df_sampled

def strategy_4_pathology_stratified(df, target_size, random_state, insurance_col):
    log_print("\n" + "=" * 80)
    log_print(f"STEP 3: SAMPLING STRATEGY 4 - STRATIFIED BY PATHOLOGY STATUS")
    log_print("=" * 80)
    log_print(f"Target: {target_size:,} samples")
    log_print(f"Available: {len(df):,} unique samples")
    log_print(f"Preserves original distribution of normal vs abnormal cases")

    log_print("\nLoading extended metadata with pathology labels...")
    meta_extended = pd.read_csv(CONFIG['METADATA_EXTENDED_PATH'])

    log_print(f"Extended metadata loaded: {len(meta_extended):,} rows")

    meta_extended['filename'] = meta_extended['dicom_path'].apply(lambda x: os.path.basename(x) if pd.notna(x) else None)
    if CONFIG['STRIP_FILE_EXTENSION']:
        meta_extended['filename'] = meta_extended['filename'].apply(lambda x: os.path.splitext(x)[0] if pd.notna(x) else None)

    df_merged = pd.merge(df, meta_extended[['filename', 'No Finding', 'Atelectasis', 'Cardiomegaly',
                                             'Consolidation', 'Edema', 'Enlarged Cardiomediastinum',
                                             'Fracture', 'Lung Lesion', 'Lung Opacity',
                                             'Pleural Effusion', 'Pleural Other', 'Pneumonia',
                                             'Pneumothorax', 'Support Devices']],
                        on='filename', how='inner')

    log_print(f"After merging with pathology data: {len(df_merged):,} rows")

    pathology_cols = ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema',
                     'Enlarged Cardiomediastinum', 'Fracture', 'Lung Lesion',
                     'Lung Opacity', 'Pleural Effusion', 'Pleural Other',
                     'Pneumonia', 'Pneumothorax', 'Support Devices']

    df_merged['has_pathology'] = df_merged[pathology_cols].fillna(0).sum(axis=1) > 0
    df_merged['pathology_status'] = df_merged['has_pathology'].apply(lambda x: 'abnormal' if x else 'normal')

    log_print(f"\nPathology status distribution:")
    for status, count in df_merged['pathology_status'].value_counts().items():
        log_print(f"   {status}: {count:,} ({count/len(df_merged)*100:.2f}%)")

    df_clean = df_merged[df_merged[insurance_col].notna()].copy()

    sampled_dfs = []
    for status in df_clean['pathology_status'].unique():
        status_df = df_clean[df_clean['pathology_status'] == status]
        n_samples = int(target_size * len(status_df) / len(df_clean))
        if len(status_df) >= n_samples:
            sampled_dfs.append(status_df.sample(n=n_samples, random_state=random_state))
        else:
            sampled_dfs.append(status_df)

    df_sampled = pd.concat(sampled_dfs, ignore_index=True)

    log_print(f"\nFinal sample size: {len(df_sampled):,}")
    log_print(f"Final insurance distribution:")
    for insurance, count in df_sampled[insurance_col].value_counts().items():
        log_print(f"   {insurance}: {count:,} ({count/len(df_sampled)*100:.2f}%)")

    return df_sampled

def strategy_5_pathology_stratified_clean(df, target_size, random_state, insurance_col):
    log_print("\n" + "=" * 80)
    log_print(f"STEP 3: SAMPLING STRATEGY 5 - STRATIFIED BY PATHOLOGY (CLEAN)")
    log_print("=" * 80)
    log_print(f"Target: {target_size:,} samples")
    log_print(f"Available: {len(df):,} unique samples")
    log_print(f"Excludes cases with only Support Devices + No Finding")

    log_print("\nLoading extended metadata with pathology labels...")
    meta_extended = pd.read_csv(CONFIG['METADATA_EXTENDED_PATH'])

    meta_extended['filename'] = meta_extended['dicom_path'].apply(lambda x: os.path.basename(x) if pd.notna(x) else None)
    if CONFIG['STRIP_FILE_EXTENSION']:
        meta_extended['filename'] = meta_extended['filename'].apply(lambda x: os.path.splitext(x)[0] if pd.notna(x) else None)

    df_merged = pd.merge(df, meta_extended[['filename', 'No Finding', 'Atelectasis', 'Cardiomegaly',
                                             'Consolidation', 'Edema', 'Enlarged Cardiomediastinum',
                                             'Fracture', 'Lung Lesion', 'Lung Opacity',
                                             'Pleural Effusion', 'Pleural Other', 'Pneumonia',
                                             'Pneumothorax', 'Support Devices']],
                        on='filename', how='inner')

    pathology_cols_no_support = ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema',
                                  'Enlarged Cardiomediastinum', 'Fracture', 'Lung Lesion',
                                  'Lung Opacity', 'Pleural Effusion', 'Pleural Other',
                                  'Pneumonia', 'Pneumothorax']

    df_merged['has_real_pathology'] = df_merged[pathology_cols_no_support].fillna(0).sum(axis=1) > 0
    df_merged['has_support_devices'] = df_merged['Support Devices'].fillna(0) > 0
    df_merged['has_no_finding'] = df_merged['No Finding'].fillna(0) > 0

    support_only_mask = (df_merged['has_support_devices']) & (df_merged['has_no_finding']) & (~df_merged['has_real_pathology'])

    log_print(f"\nExcluding {support_only_mask.sum():,} cases with only Support Devices + No Finding")
    df_filtered = df_merged[~support_only_mask].copy()

    df_filtered['pathology_status'] = df_filtered['has_real_pathology'].apply(lambda x: 'abnormal' if x else 'normal')

    df_clean = df_filtered[df_filtered[insurance_col].notna()].copy()

    sampled_dfs = []
    for status in df_clean['pathology_status'].unique():
        status_df = df_clean[df_clean['pathology_status'] == status]
        n_samples = int(target_size * len(status_df) / len(df_clean))
        if len(status_df) >= n_samples:
            sampled_dfs.append(status_df.sample(n=n_samples, random_state=random_state))
        else:
            sampled_dfs.append(status_df)

    df_sampled = pd.concat(sampled_dfs, ignore_index=True)

    log_print(f"\nFinal sample size: {len(df_sampled):,}")
    log_print(f"Final insurance distribution:")
    for insurance, count in df_sampled[insurance_col].value_counts().items():
        log_print(f"   {insurance}: {count:,} ({count/len(df_sampled)*100:.2f}%)")

    return df_sampled

# Execute selected strategy
strategy = CONFIG['SAMPLING_STRATEGY']
if strategy == 1:
    meta_sampled = strategy_1_balanced_insurance(meta_unique, CONFIG['SAMPLE_SIZE'],
                                              CONFIG['RANDOM_STATE'], CONFIG['INSURANCE_COLUMN'])
elif strategy == 2:
    meta_sampled = strategy_2_stratified_proportional(meta_unique, CONFIG['SAMPLE_SIZE'],
                                                      CONFIG['RANDOM_STATE'], CONFIG['INSURANCE_COLUMN'])
elif strategy == 3:
    meta_sampled = strategy_3_random_sampling(meta_unique, CONFIG['SAMPLE_SIZE'],
                                              CONFIG['RANDOM_STATE'], CONFIG['INSURANCE_COLUMN'])
elif strategy == 4:
    meta_sampled = strategy_4_pathology_stratified(meta_unique, CONFIG['SAMPLE_SIZE'],
                                                   CONFIG['RANDOM_STATE'], CONFIG['INSURANCE_COLUMN'])
elif strategy == 5:
    meta_sampled = strategy_5_pathology_stratified_clean(meta_unique, CONFIG['SAMPLE_SIZE'],
                                                         CONFIG['RANDOM_STATE'], CONFIG['INSURANCE_COLUMN'])
else:
    raise ValueError(f"Invalid SAMPLING_STRATEGY: {strategy}. Must be 1, 2, 3, 4, or 5.")

log_print("\n" + "=" * 80)
log_print(f"STEP 4: LOADING ONLY {len(meta_sampled):,} REQUIRED EMBEDDINGS")
log_print("=" * 80)

filenames_to_load = set(meta_sampled['filename'].values)
log_print(f"Created lookup set with {len(filenames_to_load):,} filenames")

log_print("\nOnly loading embeddings we need - this is FAST!")

records = []
batch_files = sorted(glob(CONFIG['EMBEDDING_PATH']))
log_print(f"Found {len(batch_files)} batch files")

embeddings_found = 0

for path in tqdm(batch_files, desc="Scanning batches", disable=not CONFIG['VERBOSE']):
    batch_name = os.path.basename(path)
    data = np.load(path, allow_pickle=True)

    for emb, fname in zip(data["embeddings"], data["filenames"]):
        fname_clean = str(fname)

        if fname_clean in filenames_to_load:
            records.append({
                "embedding": emb,
                "filename": fname_clean,
                "batch": batch_name
            })
            embeddings_found += 1

            if embeddings_found >= len(filenames_to_load):
                break

    if embeddings_found >= len(filenames_to_load):
        log_print(f"\nFound all {len(filenames_to_load):,} embeddings! Stopping early.")
        break

log_print(f"\nEMBEDDINGS LOADED: {len(records):,}")

if len(records) == 0:
    log_print("\nERROR: No embeddings matched!")
    save_log()
    raise ValueError("No embeddings matched. Enable DEBUG_FILENAME_MATCHING for more details.")

emb_df = pd.DataFrame(records)
log_print(f"Match rate: {len(emb_df):,} / {len(filenames_to_load):,} ({len(emb_df)/len(filenames_to_load)*100:.2f}%)")

if len(records) > 0:
    log_print(f"Embedding shape: {records[0]['embedding'].shape}")
    log_print(f"Embedding dimension (flattened): {records[0]['embedding'].flatten().shape[0]:,} features")

log_print("\n" + "=" * 80)
log_print("STEP 5: MERGING SAMPLED METADATA WITH EMBEDDINGS")
log_print("=" * 80)

df_final = pd.merge(meta_sampled, emb_df, on="filename", how="inner")

log_print(f"FINAL DATASET SIZE: {len(df_final):,} rows")

log_print(f"\nFinal insurance distribution:")
for insurance, count in df_final[CONFIG['INSURANCE_COLUMN']].value_counts().items():
    percentage = (count / len(df_final)) * 100
    log_print(f"   {insurance}: {count:,} ({percentage:.2f}%)")

# Save directly to OUTPUT_DIR (no subfolders)
output_path = os.path.join(CONFIG['OUTPUT_DIR'], OUTPUT_NAME)

log_print("\n" + "=" * 80)
log_print("STEP 6: SAVING OUTPUT (PICKLE FORMAT)")
log_print("=" * 80)

df_final.to_pickle(output_path)

log_print(f"Output saved to: {output_path}")
log_print(f"Total rows saved: {len(df_final):,}")
log_print(f"File size: {os.path.getsize(output_path) / (1024*1024):.2f} MB")

# Verification
log_print("\n" + "=" * 80)
log_print("VERIFICATION")
log_print("=" * 80)
df_verify = pd.read_pickle(output_path)
log_print(f"Loaded back {len(df_verify):,} rows")
log_print(f"Embedding shape: {df_verify['embedding'].iloc[0].shape}")

log_print("\n" + "=" * 80)
log_print("PIPELINE COMPLETE!")
log_print("=" * 80)
log_print(f"\nOutput: {output_path}")
log_print(f"Log: {log_path}")

save_log()

CHEST X-RAY INSURANCE CLASSIFICATION PIPELINE
Timestamp: 2025-12-14 10:05:03
Target Sample Size: 2,000 samples
Train/Test split: 80% / 20%
Sampling Strategy: 1 - Balanced Insurance (50/50)
Output directory: /content/drive/MyDrive/projects/Quantum-Echos/mimic-cxs_data/data-cleaned
Output file: data_type1_insurance.pkl (auto-generated based on strategy 1)

STEP 1: LOADING COMPLETE METADATA
Loaded metadata from: /content/drive/MyDrive/projects/Quantum-Echos/mimic-cxs_data/verified_dicom_clincalinfo.csv
TOTAL ROWS IN FULL DATASET: 243,334
Original columns: ['dicom_path', 'dicom_id', 'subject_id', 'study_id', 'PerformedProcedureStepDescription', 'ViewPosition', 'Rows', 'Columns', 'ProcedureCodeSequence_CodeMeaning', 'ViewCodeSequence_CodeMeaning']...

Creating filename column from dicom_path...
Stripped file extensions from filenames to match NPZ format
Created filename column from dicom_path
Rows after removing invalid paths: 243,334
Example filename: 02aa804e-bde0afdd-112c0b34-7bc16630-4e

Scanning batches:  98%|█████████▊| 48/49 [19:47<00:24, 24.73s/it]


Found all 2,000 embeddings! Stopping early.

EMBEDDINGS LOADED: 2,000
Match rate: 2,000 / 2,000 (100.00%)
Embedding shape: (8, 8, 1376)
Embedding dimension (flattened): 88,064 features

STEP 5: MERGING SAMPLED METADATA WITH EMBEDDINGS
FINAL DATASET SIZE: 2,000 rows

Final insurance distribution:
   Medicaid_Medicare: 1,000 (50.00%)
   Private: 1,000 (50.00%)

STEP 6: SAVING OUTPUT (PICKLE FORMAT)


Output saved to: /content/drive/MyDrive/projects/Quantum-Echos/mimic-cxs_data/data-cleaned/data_type1_insurance.pkl
Total rows saved: 2,000
File size: 673.17 MB

VERIFICATION
Loaded back 2,000 rows
Embedding shape: (8, 8, 1376)

PIPELINE COMPLETE!

Output: /content/drive/MyDrive/projects/Quantum-Echos/mimic-cxs_data/data-cleaned/data_type1_insurance.pkl
Log: /content/drive/MyDrive/projects/Quantum-Echos/mimic-cxs_data/data-cleaned/preprocessing_log_strategy1_insurance_20251214_100503.txt

Log saved to: /content/drive/MyDrive/projects/Quantum-Echos/mimic-cxs_data/data-cleaned/preprocessing_log_strategy1_insurance_20251214_100503.txt


This complete code includes all 5 sampling strategies:

Strategy 1: Balanced Gender (50/50)

Strategy 2: Stratified Proportional (maintains original gender distribution)

Strategy 3: Simple Random Sampling

Strategy 4: Stratifies by pathology status (normal vs abnormal) while preserving the original distribution proportions. It considers any pathology finding (including Support Devices) as abnormal.

Strategy 5: Similar to Strategy 4, but excludes cases where only "Support Devices" appears with "No Finding" (as recommended by Chi-Yu). This gives a cleaner separation between truly normal and abnormal cases.

To use different strategies, just change 'SAMPLING_STRATEGY': 1 in the CONFIG to the number you want (1-5), and update 'OUTPUT_NAME' accordingly (e.g., 'data_type4.parquet' or 'data_type5.parquet').

**IMPORTANT**: Now saves as `.parquet` instead of `.csv` to preserve full embeddings (88,064 features instead of truncated 216).

# Dataset design - multiple seeds

**NOTE**: This version uses **INSURANCE** as the label instead of gender. Insurance labels are derived from `admisions.csv`:
- **Private** → Class 1 (one-hot: [0., 1.])
- **Medicaid/Medicare** → Class 0 (one-hot: [1., 0.])

1. **Added `NUM_SEEDS` parameter** in CONFIG (set to 20)
2. **Created output folder structure** based on sampling strategy name (e.g., "data_type1", "data_type4", etc.)
3. **Load all embeddings once** at the beginning instead of loading them 20 times
4. **Loop through seeds** from 1 to 20, generating different samples for each seed
5. **Each seed gets its own file**: `data_type1_seed1.pkl`, `data_type1_seed2.pkl`, etc.
6. **Different random states** for each seed: BASE_RANDOM_STATE + seed_idx (so 43, 44, 45... up to 62)

To use it, just change `SAMPLING_STRATEGY` in CONFIG to the desired strategy (1-5), and it will automatically create the appropriate folder and 20 seed files.

**IMPORTANT**: Now saves as `.pkl` (pickle) instead of `.csv` to preserve full multi-dimensional embeddings (shape 8, 8, 1376).

In [3]:
import numpy as np
import pandas as pd
from glob import glob
import os
import re
from tqdm import tqdm
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                            roc_auc_score, confusion_matrix, classification_report)
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import gc
from datetime import datetime

CONFIG = {
    'SAMPLE_SIZE': 2000,
    'TRAIN_TEST_SPLIT_RATIO': 0.8,
    'BASE_RANDOM_STATE': 42,
    'NUM_SEEDS': 20,

    'EMBEDDING_PATH': "/content/drive/MyDrive/projects/Quantum-Echos/mimic-cxs_data/ve_cxr/elixir/batch_*.npz",
    'METADATA_PATH': '/content/drive/MyDrive/projects/Quantum-Echos/mimic-cxs_data/verified_dicom_clincalinfo.csv',
    'METADATA_EXTENDED_PATH': "/content/drive/MyDrive/projects/Quantum-Echos/mimic-cxs_data/verified_dicom_updatedlabel_wide_processed.csv",
    'PATH_TO_ADMISIONS': "/content/drive/MyDrive/projects/Quantum-Echos/mimic-cxs_data/admissions.csv",

    # Base output directory - subfolders created per strategy
    'OUTPUT_BASE_DIR': '/content/drive/MyDrive/projects/Quantum-Echos/mimic-cxs_data/data-cleaned-seeding-insurance',

    'PATIENT_ID_COLUMN': 'subject_id',
    'INSURANCE_COLUMN': 'new_insurance_type',
    'DICOM_PATH_COLUMN': 'dicom_path',

    # SAMPLING_STRATEGY determines subfolder and filename automatically
    'SAMPLING_STRATEGY': 5,

    'REMOVE_DUPLICATES': True,
    'REMOVE_DUPLICATE_FILENAMES': True,
    'ONE_IMAGE_PER_PATIENT': True,

    'STRIP_FILE_EXTENSION': True,
    'DEBUG_FILENAME_MATCHING': False,
    'VERBOSE': True,

    # Resume functionality
    'RESUME_FROM_EXISTING': True,  # Set to False to force restart from seed 1
}

SAMPLING_STRATEGIES = {
    1: {
        'name': 'Balanced Insurance (50/50)',
        'description': 'Ensures equal distribution across insurance types (Private vs Medicaid/Medicare)',
        'function': 'strategy_1_balanced_insurance'
    },
    2: {
        'name': 'Stratified Proportional',
        'description': 'Maintains original insurance proportions',
        'function': 'strategy_2_stratified_proportional'
    },
    3: {
        'name': 'Simple Random Sampling',
        'description': 'Pure random selection without stratification',
        'function': 'strategy_3_random_sampling'
    },
    4: {
        'name': 'Stratified by Pathology Status',
        'description': 'Preserves original distribution of normal vs abnormal cases',
        'function': 'strategy_4_pathology_stratified'
    },
    5: {
        'name': 'Stratified by Pathology Status (Excluding Support Devices Only)',
        'description': 'Preserves distribution, excludes cases with only Support Devices and No Finding',
        'function': 'strategy_5_pathology_stratified_clean'
    }
}

# Auto-generate output folder and filename prefix based on sampling strategy
STRATEGY_FOLDER = f"data_type{CONFIG['SAMPLING_STRATEGY']}"
OUTPUT_PREFIX = f"data_type{CONFIG['SAMPLING_STRATEGY']}_insurance"

# Create output directory structure: base_dir/data_typeX/
OUTPUT_DIR = os.path.join(CONFIG['OUTPUT_BASE_DIR'], STRATEGY_FOLDER)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Setup logging to file
log_messages = []
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_filename = f"preprocessing_log_multipleseeds_type{CONFIG['SAMPLING_STRATEGY']}_{timestamp}.txt"
log_path = os.path.join(OUTPUT_DIR, log_filename)

def log_print(message, end='\n'):
    """Print message and store for later saving to file"""
    log_messages.append(message)
    if CONFIG['VERBOSE']:
        print(message, end=end)

def save_log():
    """Save all log messages to file"""
    with open(log_path, 'w') as f:
        f.write('\n'.join(log_messages))
    print(f"\nLog saved to: {log_path}")


# ============================================================================
# RESUME FUNCTIONALITY: Detect existing seed files
# ============================================================================
def detect_existing_seeds(output_dir, output_prefix):
    """
    Scan the output directory for existing seed files and return a set of
    already completed seed indices.

    Returns:
        tuple: (set of completed seed indices, list of existing files with details)
    """
    completed_seeds = set()
    existing_files = []

    # Pattern to match seed files: data_type4_insurance_seed1.pkl, etc.
    pattern = re.compile(rf'^{re.escape(output_prefix)}_seed(\d+)\.pkl$')

    if not os.path.exists(output_dir):
        return completed_seeds, existing_files

    for filename in os.listdir(output_dir):
        match = pattern.match(filename)
        if match:
            seed_idx = int(match.group(1))
            filepath = os.path.join(output_dir, filename)
            file_size = os.path.getsize(filepath)

            # Only count files that are non-empty (at least 1KB to be safe)
            if file_size > 1024:
                completed_seeds.add(seed_idx)
                existing_files.append({
                    'seed_idx': seed_idx,
                    'filename': filename,
                    'size_mb': file_size / (1024 * 1024),
                    'modified': datetime.fromtimestamp(os.path.getmtime(filepath))
                })

    # Sort by seed index for display
    existing_files.sort(key=lambda x: x['seed_idx'])

    return completed_seeds, existing_files


def get_seeds_to_process(num_seeds, completed_seeds, resume_enabled):
    """
    Determine which seeds need to be processed.

    Args:
        num_seeds: Total number of seeds to generate
        completed_seeds: Set of already completed seed indices
        resume_enabled: Whether resume functionality is enabled

    Returns:
        list: Seed indices that need to be processed
    """
    all_seeds = set(range(1, num_seeds + 1))

    if resume_enabled and completed_seeds:
        seeds_to_process = sorted(all_seeds - completed_seeds)
    else:
        seeds_to_process = sorted(all_seeds)

    return seeds_to_process


# ============================================================================
# DETECT EXISTING PROGRESS
# ============================================================================
log_print("=" * 80)
log_print("CHEST X-RAY INSURANCE CLASSIFICATION PIPELINE - MULTIPLE SEEDS (RESUMABLE)")
log_print("=" * 80)
log_print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Check for existing seeds before anything else
completed_seeds, existing_files = detect_existing_seeds(OUTPUT_DIR, OUTPUT_PREFIX)

log_print(f"\n{'=' * 60}")
log_print("RESUME STATUS CHECK")
log_print(f"{'=' * 60}")
log_print(f"Output directory: {OUTPUT_DIR}")
log_print(f"Resume from existing: {CONFIG['RESUME_FROM_EXISTING']}")

if existing_files:
    log_print(f"\nFound {len(completed_seeds)} existing seed file(s):")
    for f in existing_files:
        log_print(f"   Seed {f['seed_idx']:2d}: {f['filename']} ({f['size_mb']:.2f} MB, modified {f['modified'].strftime('%Y-%m-%d %H:%M')})")
else:
    log_print("\nNo existing seed files found. Starting fresh.")

# Determine which seeds to process
seeds_to_process = get_seeds_to_process(CONFIG['NUM_SEEDS'], completed_seeds, CONFIG['RESUME_FROM_EXISTING'])

if not seeds_to_process:
    log_print(f"\n*** ALL {CONFIG['NUM_SEEDS']} SEEDS ALREADY COMPLETED! ***")
    log_print("Nothing to do. Set RESUME_FROM_EXISTING=False to force regeneration.")
    save_log()
    exit(0)

log_print(f"\nSeeds to process: {len(seeds_to_process)} of {CONFIG['NUM_SEEDS']}")
if CONFIG['RESUME_FROM_EXISTING'] and completed_seeds:
    log_print(f"   Skipping already completed: {sorted(completed_seeds)}")
log_print(f"   Will process: {seeds_to_process}")

log_print(f"\n{'=' * 60}")
log_print("CONFIGURATION")
log_print(f"{'=' * 60}")
log_print(f"Target Sample Size: {CONFIG['SAMPLE_SIZE']:,} samples")
log_print(f"Number of Seeds (total): {CONFIG['NUM_SEEDS']}")
log_print(f"Number of Seeds (remaining): {len(seeds_to_process)}")
log_print(f"Base Random State: {CONFIG['BASE_RANDOM_STATE']}")
log_print(f"Sampling Strategy: {CONFIG['SAMPLING_STRATEGY']} - {SAMPLING_STRATEGIES[CONFIG['SAMPLING_STRATEGY']]['name']}")
log_print(f"\nOutput Structure:")
log_print(f"  Base Directory: {CONFIG['OUTPUT_BASE_DIR']}")
log_print(f"  Strategy Folder: {STRATEGY_FOLDER}/")
log_print(f"  Full Output Path: {OUTPUT_DIR}")
log_print("=" * 80)

log_print("\n" + "=" * 80)
log_print("STEP 1: LOADING COMPLETE METADATA")
log_print("=" * 80)

meta = pd.read_csv(CONFIG['METADATA_PATH'])

log_print(f"Loaded metadata from: {CONFIG['METADATA_PATH']}")
log_print(f"TOTAL ROWS IN FULL DATASET: {len(meta):,}")

log_print("\nCreating filename column from dicom_path...")
meta['filename'] = meta[CONFIG['DICOM_PATH_COLUMN']].apply(lambda x: os.path.basename(x) if pd.notna(x) else None)

if CONFIG['STRIP_FILE_EXTENSION']:
    meta['filename'] = meta['filename'].apply(lambda x: os.path.splitext(x)[0] if pd.notna(x) else None)
    log_print("Stripped file extensions from filenames to match NPZ format")

meta = meta[meta['filename'].notna()].reset_index(drop=True)
log_print(f"Rows after removing invalid paths: {len(meta):,}")

# ============================================================================
# STEP 1.5: LOADING AND PREPROCESSING ADMISSIONS DATA FOR INSURANCE
# ============================================================================
log_print("\n" + "=" * 80)
log_print("STEP 1.5: LOADING AND PREPROCESSING ADMISSIONS DATA FOR INSURANCE")
log_print("=" * 80)

admissions = pd.read_csv(CONFIG['PATH_TO_ADMISIONS'])
log_print(f"Loaded admissions from: {CONFIG['PATH_TO_ADMISIONS']}")
log_print(f"Total admissions rows: {len(admissions):,}")
log_print(f"Admissions columns: {list(admissions.columns)}")

# ============================================================================
# INSURANCE PREPROCESSING - DETAILED LOGGING
# ============================================================================
log_print("\n" + "-" * 60)
log_print("INSURANCE COLUMN ANALYSIS (RAW DATA)")
log_print("-" * 60)

raw_insurance_counts = admissions['insurance'].value_counts(dropna=False)
log_print(f"\nRaw insurance values in admissions.csv:")
for ins_val, count in raw_insurance_counts.items():
    percentage = (count / len(admissions)) * 100
    log_print(f"   '{ins_val}': {count:,} ({percentage:.2f}%)")

log_print(f"\nTotal unique raw insurance values: {admissions['insurance'].nunique()}")
log_print(f"Missing/NaN insurance values: {admissions['insurance'].isna().sum():,}")

# ============================================================================
# INSURANCE MAPPING LOGIC (MATCHING REFERENCE NOTEBOOK)
# ============================================================================
log_print("\n" + "-" * 60)
log_print("INSURANCE MAPPING LOGIC")
log_print("-" * 60)
log_print("\nMapping rules (same as reference MIMIC_raw dataset class):")
log_print("   'Private'                    -> 'Private'        (Class 1: one-hot [0., 1.])")
log_print("   'Medicaid' or 'Medicare'     -> 'Medicaid_Medicare' (Class 0: one-hot [1., 0.])")
log_print("   Other/Unknown/NaN            -> EXCLUDED (set to None)")

def map_insurance_type(insurance):
    """
    Map insurance values to binary categories matching reference notebook.
    """
    if pd.isna(insurance):
        return None
    insurance_str = str(insurance).strip()
    if insurance_str.lower() == 'private':
        return 'Private'
    elif insurance_str.lower() in ['medicaid', 'medicare']:
        return 'Medicaid_Medicare'
    else:
        return None

admissions['new_insurance_type'] = admissions['insurance'].apply(map_insurance_type)

# ============================================================================
# POST-MAPPING ANALYSIS
# ============================================================================
log_print("\n" + "-" * 60)
log_print("POST-MAPPING ANALYSIS")
log_print("-" * 60)

excluded_count = admissions['new_insurance_type'].isna().sum()
included_count = admissions['new_insurance_type'].notna().sum()
log_print(f"\nRecords after mapping:")
log_print(f"   Included (valid insurance): {included_count:,} ({included_count/len(admissions)*100:.2f}%)")
log_print(f"   Excluded (other/unknown):   {excluded_count:,} ({excluded_count/len(admissions)*100:.2f}%)")

excluded_values = admissions[admissions['new_insurance_type'].isna()]['insurance'].value_counts(dropna=False)
if len(excluded_values) > 0:
    log_print(f"\nExcluded raw insurance values:")
    for val, count in excluded_values.items():
        log_print(f"   '{val}': {count:,} records excluded")

admissions_clean = admissions[admissions['new_insurance_type'].notna()][['subject_id', 'new_insurance_type']].copy()
admissions_clean = admissions_clean.drop_duplicates(subset=['subject_id'], keep='first')

# ============================================================================
# FINAL CLASS DISTRIBUTION
# ============================================================================
log_print("\n" + "-" * 60)
log_print("FINAL INSURANCE CLASS DISTRIBUTION")
log_print("-" * 60)

num_classes = admissions_clean['new_insurance_type'].nunique()
class_names = sorted(admissions_clean['new_insurance_type'].unique())
log_print(f"\nNumber of classes: {num_classes}")
log_print(f"Class names: {class_names}")

log_print(f"\nUnique patients with valid insurance: {len(admissions_clean):,}")
ins_counts = admissions_clean['new_insurance_type'].value_counts()
for ins_type, count in ins_counts.items():
    percentage = (count / len(admissions_clean)) * 100
    log_print(f"   {ins_type}: {count:,} ({percentage:.2f}%)")

# ============================================================================
# ONE-HOT ENCODING REFERENCE
# ============================================================================
log_print("\n" + "-" * 60)
log_print("ONE-HOT ENCODING REFERENCE (for model training)")
log_print("-" * 60)
log_print("\nEncoding scheme (matching reference notebook MIMIC_raw class):")
log_print("   'Private'          -> torch.Tensor([0., 1.])  # Class 1")
log_print("   'Medicaid_Medicare' -> torch.Tensor([1., 0.])  # Class 0")

# ============================================================================
# MERGE WITH METADATA
# ============================================================================
log_print("\n" + "-" * 60)
log_print("MERGING WITH IMAGE METADATA")
log_print("-" * 60)

meta_before_merge = len(meta)
meta = pd.merge(meta, admissions_clean, on='subject_id', how='inner')
log_print(f"\nMetadata rows before merge: {meta_before_merge:,}")
log_print(f"Metadata rows after merge:  {len(meta):,}")
log_print(f"Rows lost in merge: {meta_before_merge - len(meta):,} (patients without valid insurance)")

log_print(f"\nInsurance distribution in FULL merged dataset:")
insurance_counts = meta[CONFIG['INSURANCE_COLUMN']].value_counts()
for insurance, count in insurance_counts.items():
    percentage = (count / len(meta)) * 100
    log_print(f"   {insurance}: {count:,} ({percentage:.2f}%)")

unique_patients = meta[CONFIG['PATIENT_ID_COLUMN']].nunique()
log_print(f"\nTotal unique patients: {unique_patients:,}")

def remove_duplicates(df, patient_col, random_state):
    log_print("\n" + "=" * 80)
    log_print("STEP 2: REMOVING DUPLICATES - KEEPING ONLY UNIQUE DATA")
    log_print("=" * 80)

    original_size = len(df)
    log_print(f"Original dataset size: {original_size:,}")

    if CONFIG['REMOVE_DUPLICATE_FILENAMES']:
        df_unique = df.drop_duplicates(subset=['filename'], keep='first')
        log_print(f"After removing duplicate filenames: {len(df_unique):,} rows")
    else:
        df_unique = df.copy()

    if CONFIG['ONE_IMAGE_PER_PATIENT']:
        df_unique = df_unique.groupby(patient_col).sample(n=1, random_state=random_state).reset_index(drop=True)
        log_print(f"After keeping one image per patient: {len(df_unique):,} rows")

    log_print(f"\nInsurance distribution in unique dataset:")
    for insurance, count in df_unique[CONFIG['INSURANCE_COLUMN']].value_counts().items():
        log_print(f"   {insurance}: {count:,} ({count/len(df_unique)*100:.2f}%)")

    return df_unique

if CONFIG['REMOVE_DUPLICATES']:
    meta_unique = remove_duplicates(meta, CONFIG['PATIENT_ID_COLUMN'], CONFIG['BASE_RANDOM_STATE'])
else:
    meta_unique = meta.copy()

def strategy_1_balanced_insurance(df, target_size, random_state, insurance_col):
    log_print(f"\nSTRATEGY 1 - BALANCED INSURANCE (50/50) - Seed {random_state}")

    df_clean = df[df[insurance_col].notna()].copy()
    samples_per_insurance = target_size // 2
    sampled_dfs = []

    for insurance_value in sorted(df_clean[insurance_col].unique()):
        insurance_df = df_clean[df_clean[insurance_col] == insurance_value]
        if len(insurance_df) >= samples_per_insurance:
            sampled_dfs.append(insurance_df.sample(n=samples_per_insurance, random_state=random_state))
        else:
            sampled_dfs.append(insurance_df)

    df_sampled = pd.concat(sampled_dfs, ignore_index=True)
    insurance_dist = " | ".join([f"{ins}={cnt:,}" for ins, cnt in df_sampled[insurance_col].value_counts().items()])
    log_print(f"Sample size: {len(df_sampled):,} ({insurance_dist})")

    return df_sampled

def strategy_2_stratified_proportional(df, target_size, random_state, insurance_col):
    log_print(f"\nSTRATEGY 2 - STRATIFIED PROPORTIONAL - Seed {random_state}")

    df_clean = df[df[insurance_col].notna()].copy()
    sampled_dfs = []
    for insurance_value in df_clean[insurance_col].unique():
        insurance_df = df_clean[df_clean[insurance_col] == insurance_value]
        n_samples = int(target_size * len(insurance_df) / len(df_clean))
        if len(insurance_df) >= n_samples:
            sampled_dfs.append(insurance_df.sample(n=n_samples, random_state=random_state))
        else:
            sampled_dfs.append(insurance_df)

    df_sampled = pd.concat(sampled_dfs, ignore_index=True)
    insurance_dist = " | ".join([f"{ins}={cnt:,}" for ins, cnt in df_sampled[insurance_col].value_counts().items()])
    log_print(f"Sample size: {len(df_sampled):,} ({insurance_dist})")

    return df_sampled

def strategy_3_random_sampling(df, target_size, random_state, insurance_col):
    log_print(f"\nSTRATEGY 3 - SIMPLE RANDOM SAMPLING - Seed {random_state}")

    df_clean = df[df[insurance_col].notna()].copy()
    if len(df_clean) >= target_size:
        df_sampled = df_clean.sample(n=target_size, random_state=random_state).reset_index(drop=True)
    else:
        df_sampled = df_clean

    insurance_dist = " | ".join([f"{ins}={cnt:,}" for ins, cnt in df_sampled[insurance_col].value_counts().items()])
    log_print(f"Sample size: {len(df_sampled):,} ({insurance_dist})")

    return df_sampled

def strategy_4_pathology_stratified(df, target_size, random_state, insurance_col):
    log_print(f"\nSTRATEGY 4 - STRATIFIED BY PATHOLOGY STATUS - Seed {random_state}")

    meta_extended = pd.read_csv(CONFIG['METADATA_EXTENDED_PATH'])
    meta_extended['filename'] = meta_extended['dicom_path'].apply(lambda x: os.path.basename(x) if pd.notna(x) else None)
    if CONFIG['STRIP_FILE_EXTENSION']:
        meta_extended['filename'] = meta_extended['filename'].apply(lambda x: os.path.splitext(x)[0] if pd.notna(x) else None)

    df_merged = pd.merge(df, meta_extended[['filename', 'No Finding', 'Atelectasis', 'Cardiomegaly',
                                             'Consolidation', 'Edema', 'Enlarged Cardiomediastinum',
                                             'Fracture', 'Lung Lesion', 'Lung Opacity',
                                             'Pleural Effusion', 'Pleural Other', 'Pneumonia',
                                             'Pneumothorax', 'Support Devices']],
                        on='filename', how='inner')

    pathology_cols = ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema',
                     'Enlarged Cardiomediastinum', 'Fracture', 'Lung Lesion',
                     'Lung Opacity', 'Pleural Effusion', 'Pleural Other',
                     'Pneumonia', 'Pneumothorax', 'Support Devices']

    df_merged['has_pathology'] = df_merged[pathology_cols].fillna(0).sum(axis=1) > 0
    df_merged['pathology_status'] = df_merged['has_pathology'].apply(lambda x: 'abnormal' if x else 'normal')

    df_clean = df_merged[df_merged[insurance_col].notna()].copy()

    sampled_dfs = []
    for status in df_clean['pathology_status'].unique():
        status_df = df_clean[df_clean['pathology_status'] == status]
        n_samples = int(target_size * len(status_df) / len(df_clean))
        if len(status_df) >= n_samples:
            sampled_dfs.append(status_df.sample(n=n_samples, random_state=random_state))
        else:
            sampled_dfs.append(status_df)

    df_sampled = pd.concat(sampled_dfs, ignore_index=True)
    path_dist = " | ".join([f"{s}={c:,}" for s, c in df_sampled['pathology_status'].value_counts().items()])
    log_print(f"Sample size: {len(df_sampled):,} ({path_dist})")

    return df_sampled

def strategy_5_pathology_stratified_clean(df, target_size, random_state, insurance_col):
    log_print(f"\nSTRATEGY 5 - STRATIFIED BY PATHOLOGY (CLEAN) - Seed {random_state}")

    meta_extended = pd.read_csv(CONFIG['METADATA_EXTENDED_PATH'])
    meta_extended['filename'] = meta_extended['dicom_path'].apply(lambda x: os.path.basename(x) if pd.notna(x) else None)
    if CONFIG['STRIP_FILE_EXTENSION']:
        meta_extended['filename'] = meta_extended['filename'].apply(lambda x: os.path.splitext(x)[0] if pd.notna(x) else None)

    df_merged = pd.merge(df, meta_extended[['filename', 'No Finding', 'Atelectasis', 'Cardiomegaly',
                                             'Consolidation', 'Edema', 'Enlarged Cardiomediastinum',
                                             'Fracture', 'Lung Lesion', 'Lung Opacity',
                                             'Pleural Effusion', 'Pleural Other', 'Pneumonia',
                                             'Pneumothorax', 'Support Devices']],
                        on='filename', how='inner')

    pathology_cols_no_support = ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema',
                                  'Enlarged Cardiomediastinum', 'Fracture', 'Lung Lesion',
                                  'Lung Opacity', 'Pleural Effusion', 'Pleural Other',
                                  'Pneumonia', 'Pneumothorax']

    df_merged['has_real_pathology'] = df_merged[pathology_cols_no_support].fillna(0).sum(axis=1) > 0
    df_merged['has_support_devices'] = df_merged['Support Devices'].fillna(0) > 0
    df_merged['has_no_finding'] = df_merged['No Finding'].fillna(0) > 0

    support_only_mask = (df_merged['has_support_devices']) & (df_merged['has_no_finding']) & (~df_merged['has_real_pathology'])
    df_filtered = df_merged[~support_only_mask].copy()

    df_filtered['pathology_status'] = df_filtered['has_real_pathology'].apply(lambda x: 'abnormal' if x else 'normal')
    df_clean = df_filtered[df_filtered[insurance_col].notna()].copy()

    sampled_dfs = []
    for status in df_clean['pathology_status'].unique():
        status_df = df_clean[df_clean['pathology_status'] == status]
        n_samples = int(target_size * len(status_df) / len(df_clean))
        if len(status_df) >= n_samples:
            sampled_dfs.append(status_df.sample(n=n_samples, random_state=random_state))
        else:
            sampled_dfs.append(status_df)

    df_sampled = pd.concat(sampled_dfs, ignore_index=True)
    path_dist = " | ".join([f"{s}={c:,}" for s, c in df_sampled['pathology_status'].value_counts().items()])
    log_print(f"Sample size: {len(df_sampled):,} ({path_dist})")

    return df_sampled

def load_embeddings_for_filenames(filenames_set):
    """Load only the embeddings needed for the given filenames"""
    records = []
    batch_files = sorted(glob(CONFIG['EMBEDDING_PATH']))
    embeddings_found = 0

    for path in tqdm(batch_files, desc="Loading embeddings", disable=not CONFIG['VERBOSE']):
        data = np.load(path, allow_pickle=True)

        for emb, fname in zip(data["embeddings"], data["filenames"]):
            fname_clean = str(fname)

            if fname_clean in filenames_set:
                records.append({
                    "embedding": emb,
                    "filename": fname_clean
                })
                embeddings_found += 1

                if embeddings_found >= len(filenames_set):
                    break

        if embeddings_found >= len(filenames_set):
            break

    return pd.DataFrame(records)


# ============================================================================
# LOAD EXISTING SUMMARY (for appending) OR CREATE NEW
# ============================================================================
def load_or_create_summary(output_dir, output_prefix, completed_seeds):
    """
    Load existing summary CSV if resuming, or prepare empty list for new run.
    """
    summary_path = os.path.join(output_dir, f"summary_{output_prefix}.csv")

    if os.path.exists(summary_path) and completed_seeds:
        try:
            existing_summary = pd.read_csv(summary_path)
            log_print(f"\nLoaded existing summary with {len(existing_summary)} entries")
            return existing_summary.to_dict('records')
        except Exception as e:
            log_print(f"\nWarning: Could not load existing summary: {e}")
            return []

    return []


# ============================================================================
# MAIN PROCESSING LOOP (only processes remaining seeds)
# ============================================================================
log_print("\n" + "=" * 80)
log_print(f"GENERATING SAMPLES FOR {len(seeds_to_process)} REMAINING SEEDS")
log_print("=" * 80)

# Load existing summary stats if resuming
summary_stats = load_or_create_summary(OUTPUT_DIR, OUTPUT_PREFIX, completed_seeds)

processed_count = 0
total_to_process = len(seeds_to_process)

for seed_idx in seeds_to_process:
    processed_count += 1
    current_seed = CONFIG['BASE_RANDOM_STATE'] + seed_idx

    log_print("\n" + "-" * 60)
    log_print(f"SEED {seed_idx}/{CONFIG['NUM_SEEDS']} (Progress: {processed_count}/{total_to_process}, Random State: {current_seed})")
    log_print("-" * 60)

    strategy = CONFIG['SAMPLING_STRATEGY']
    if strategy == 1:
        meta_sampled = strategy_1_balanced_insurance(meta_unique, CONFIG['SAMPLE_SIZE'],
                                                  current_seed, CONFIG['INSURANCE_COLUMN'])
    elif strategy == 2:
        meta_sampled = strategy_2_stratified_proportional(meta_unique, CONFIG['SAMPLE_SIZE'],
                                                          current_seed, CONFIG['INSURANCE_COLUMN'])
    elif strategy == 3:
        meta_sampled = strategy_3_random_sampling(meta_unique, CONFIG['SAMPLE_SIZE'],
                                                  current_seed, CONFIG['INSURANCE_COLUMN'])
    elif strategy == 4:
        meta_sampled = strategy_4_pathology_stratified(meta_unique, CONFIG['SAMPLE_SIZE'],
                                                       current_seed, CONFIG['INSURANCE_COLUMN'])
    elif strategy == 5:
        meta_sampled = strategy_5_pathology_stratified_clean(meta_unique, CONFIG['SAMPLE_SIZE'],
                                                             current_seed, CONFIG['INSURANCE_COLUMN'])
    else:
        raise ValueError(f"Invalid SAMPLING_STRATEGY: {strategy}. Must be 1, 2, 3, 4, or 5.")

    filenames_to_load = set(meta_sampled['filename'].values)
    log_print(f"Loading {len(filenames_to_load):,} embeddings...")

    emb_df = load_embeddings_for_filenames(filenames_to_load)

    log_print(f"Matched {len(emb_df):,} embeddings ({len(emb_df)/len(filenames_to_load)*100:.2f}%)")

    if len(emb_df) == 0:
        log_print(f"WARNING: No embeddings matched for seed {seed_idx}!")
        continue

    df_final = pd.merge(meta_sampled, emb_df, on="filename", how="inner")

    # Save to strategy-specific subfolder with auto-indexed filename
    output_filename = f"{OUTPUT_PREFIX}_seed{seed_idx}.pkl"
    output_path = os.path.join(OUTPUT_DIR, output_filename)

    df_final.to_pickle(output_path)

    file_size_mb = os.path.getsize(output_path) / (1024*1024)
    log_print(f"Saved: {STRATEGY_FOLDER}/{output_filename} ({len(df_final):,} rows, {file_size_mb:.2f} MB)")

    # Track stats
    insurance_dist = df_final[CONFIG['INSURANCE_COLUMN']].value_counts().to_dict()
    summary_stats.append({
        'seed': seed_idx,
        'random_state': current_seed,
        'samples': len(df_final),
        'file_size_mb': file_size_mb,
        **{f'insurance_{k}': v for k, v in insurance_dist.items()}
    })

    # Verify embedding shape on first processed seed
    if processed_count == 1:
        log_print(f"Embedding shape preserved: {df_final['embedding'].iloc[0].shape}")
        log_print(f"Embedding size: {df_final['embedding'].iloc[0].flatten().shape[0]:,} features")

    del emb_df, df_final, meta_sampled
    gc.collect()

# Save updated summary CSV (includes both old and new entries)
log_print("\n" + "=" * 80)
log_print("SAVING SUMMARY REPORT")
log_print("=" * 80)

summary_df = pd.DataFrame(summary_stats)
# Sort by seed for cleaner output
summary_df = summary_df.sort_values('seed').reset_index(drop=True)
summary_path = os.path.join(OUTPUT_DIR, f"summary_{OUTPUT_PREFIX}.csv")
summary_df.to_csv(summary_path, index=False)
log_print(f"Summary saved to: {STRATEGY_FOLDER}/summary_{OUTPUT_PREFIX}.csv")
log_print(f"Total entries in summary: {len(summary_df)}")

log_print("\n" + "=" * 80)
log_print("PROCESSING COMPLETE!")
log_print("=" * 80)

# Final status
final_completed, final_files = detect_existing_seeds(OUTPUT_DIR, OUTPUT_PREFIX)
log_print(f"\nFinal Status:")
log_print(f"   Total seeds completed: {len(final_completed)}/{CONFIG['NUM_SEEDS']}")
log_print(f"   Seeds processed this run: {processed_count}")
if len(final_completed) < CONFIG['NUM_SEEDS']:
    missing = set(range(1, CONFIG['NUM_SEEDS'] + 1)) - final_completed
    log_print(f"   Missing seeds: {sorted(missing)}")
else:
    log_print("   All seeds successfully generated!")

log_print(f"\nOutput Structure:")
log_print(f"  {CONFIG['OUTPUT_BASE_DIR']}/")
log_print(f"    {STRATEGY_FOLDER}/")
for seed_idx in range(1, min(4, CONFIG['NUM_SEEDS'] + 1)):
    log_print(f"      {OUTPUT_PREFIX}_seed{seed_idx}.pkl")
if CONFIG['NUM_SEEDS'] > 3:
    log_print(f"      ...")
    log_print(f"      {OUTPUT_PREFIX}_seed{CONFIG['NUM_SEEDS']}.pkl")
log_print(f"      summary_{OUTPUT_PREFIX}.csv")
log_print(f"      {log_filename}")

save_log()

CHEST X-RAY INSURANCE CLASSIFICATION PIPELINE - MULTIPLE SEEDS (RESUMABLE)
Timestamp: 2025-12-17 07:38:01

RESUME STATUS CHECK
Output directory: /content/drive/MyDrive/projects/Quantum-Echos/mimic-cxs_data/data-cleaned-seeding-insurance/data_type5
Resume from existing: True

Found 20 existing seed file(s):
   Seed  1: data_type5_insurance_seed1.pkl (673.05 MB, modified 2025-12-15 07:48)
   Seed  2: data_type5_insurance_seed2.pkl (673.05 MB, modified 2025-12-15 08:15)
   Seed  3: data_type5_insurance_seed3.pkl (673.05 MB, modified 2025-12-15 08:42)
   Seed  4: data_type5_insurance_seed4.pkl (673.05 MB, modified 2025-12-15 09:09)
   Seed  5: data_type5_insurance_seed5.pkl (673.05 MB, modified 2025-12-15 09:35)
   Seed  6: data_type5_insurance_seed6.pkl (673.05 MB, modified 2025-12-15 10:02)
   Seed  7: data_type5_insurance_seed7.pkl (673.05 MB, modified 2025-12-15 10:28)
   Seed  8: data_type5_insurance_seed8.pkl (673.05 MB, modified 2025-12-15 10:55)
   Seed  9: data_type5_insurance_see